# 📓 Lab 02 — MLP para clasificación no lineal (`make_moons`)

**Curso:** Deep Learning: de los fundamentos a los Transformers · **Sesión 1 · Laboratorio 1**
**Material del aula:** [Sesión 1 — Fundamentos](../sesiones/01-fundamentos.md) ·
**Config:** [`configs/mlp.yaml`](../configs/mlp.yaml)

### Pregunta experimental

> ¿Cómo cambia la frontera de decisión al aumentar la capacidad de una MLP
> y qué evidencia indica overfitting?

El flujo completo: datos → splits sin leakage → modelo → training loop con
early stopping → métricas → frontera de decisión → conclusión.

In [ ]:
from __future__ import annotations

import sys
sys.path.append('..')   # para importar src/ desde notebooks/

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

# Reutilizamos las piezas del curso (código comentado en src/)
from src.utils import seed_everything, detectar_dispositivo
from src.data import cargar_moons
from src.models import MoonMLP
from src.train import entrenar, run_epoch

seed_everything(42)                 # reproducibilidad ANTES de todo
device = detectar_dispositivo()
print('Dispositivo:', device)

## 1. Datos y splits

`cargar_moons` (ver [`src/data.py`](../src/data.py)) hace el trabajo delicado:
- split estratificado 70/15/15 (*estratificado*: misma proporción de clases en cada split),
- `StandardScaler` (resta la media y divide por la desviación estándar) ajustado **solo con train** (regla anti-leakage: que ninguna información de test se filtre al entrenamiento — Sesión 1 §7),
- labels con shape `(N, 1)` float32 para `BCEWithLogitsLoss`.

In [ ]:
train_loader, val_loader, test_loader, scaler = cargar_moons(
    n_samples=1500, noise=0.22, batch_size=64, seed=42
)

# Inspeccionar un batch: SIEMPRE verificar shapes antes de entrenar
x_batch, y_batch = next(iter(train_loader))
print('x_batch:', x_batch.shape, x_batch.dtype)   # (64, 2)
print('y_batch:', y_batch.shape, y_batch.dtype)   # (64, 1)

In [ ]:
# Visualizar el dataset: dos "lunas" entrelazadas — imposible
# separarlas con una línea recta. Por eso necesitamos no-linealidad.
X_visual = torch.cat([x for x, _ in train_loader]).numpy()
y_visual = torch.cat([y for _, y in train_loader]).numpy().ravel()

plt.figure(figsize=(6, 5))
plt.scatter(X_visual[:, 0], X_visual[:, 1], c=y_visual, cmap='RdBu_r',
            edgecolors='k', s=20, linewidths=0.4)
plt.title('make_moons (train, estandarizado)')
plt.xlabel('x₁'); plt.ylabel('x₂')
plt.show()

## 2. Modelo y entrenamiento

La arquitectura está en [`src/models.py → MoonMLP`](../src/models.py):
`2 → hidden → hidden → 1 logit`, con ReLU y dropout.

El training loop del curso ([`src/train.py`](../src/train.py)) implementa la secuencia
canónica **forward → loss → zero_grad → backward → step** con early stopping.

In [ ]:
model = MoonMLP(hidden_dim=32, dropout=0.10)

criterion = nn.BCEWithLogitsLoss()   # sigmoid + BCE, numéricamente estable
# AdamW: variante adaptativa del gradient descent (Sesión 2 §5);
# weight_decay penaliza pesos grandes (regularización L2)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

model, historia = entrenar(
    model, train_loader, val_loader, criterion, optimizer, device,
    max_epochs=300,
    patience=20,        # early stopping: epochs sin mejora en val antes de parar
    binaria=True,
    verbose_cada=25,
)

## 3. Evidencia 1 — Curvas de aprendizaje

¿Se cruzan las curvas? ¿Hay brecha creciente (overfitting)?
Compara con los tres regímenes de la [Sesión 1](../sesiones/01-fundamentos.md#7-generalización-la-única-cosa-que-de-verdad-importa).

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(historia.train_loss, label='train')
ax1.plot(historia.val_loss, label='validation')
ax1.set(xlabel='epoch', ylabel='BCE loss', title='Curvas de pérdida')
ax1.legend()

ax2.plot(historia.train_acc, label='train')
ax2.plot(historia.val_acc, label='validation')
ax2.set(xlabel='epoch', ylabel='accuracy', title='Curvas de accuracy')
ax2.legend()
plt.show()

## 4. Evidencia 2 — Métricas en test (una sola vez)

El test set se toca UNA vez, al final, con el mejor checkpoint restaurado.

In [ ]:
from src.evaluate import predecir, reporte_completo

test_metrics = run_epoch(model, test_loader, criterion, device, binaria=True)
print(f'Test: loss={test_metrics.loss:.4f}  accuracy={test_metrics.accuracy:.3f}\n')

y_real, y_pred, confianzas = predecir(model, test_loader, device, binaria=True)
reporte_completo(y_real, y_pred, nombres_clases=['luna 0', 'luna 1'])

## 5. Evidencia 3 — Frontera de decisión

Evaluamos el modelo sobre una malla densa del plano y coloreamos por
probabilidad: la línea negra es la frontera P=0.5.

In [ ]:
x_min, x_max = X_visual[:, 0].min() - 0.7, X_visual[:, 0].max() + 0.7
y_min, y_max = X_visual[:, 1].min() - 0.7, X_visual[:, 1].max() + 0.7
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))
malla = np.c_[xx.ravel(), yy.ravel()]

model.eval()
with torch.inference_mode():
    malla_t = torch.tensor(malla, dtype=torch.float32, device=device)
    prob = torch.sigmoid(model(malla_t)).cpu().numpy().reshape(xx.shape)

plt.figure(figsize=(7, 6))
plt.contourf(xx, yy, prob, levels=20, cmap='RdBu_r', alpha=0.65)
plt.contour(xx, yy, prob, levels=[0.5], colors='k', linewidths=2)
plt.scatter(X_visual[:, 0], X_visual[:, 1], c=y_visual, cmap='RdBu_r',
            edgecolors='k', s=20, linewidths=0.4)
plt.title('Frontera de decisión de la MLP')
plt.xlabel('x₁ estandarizada'); plt.ylabel('x₂ estandarizada')
plt.show()

## 6. 🧪 Experimentos obligatorios

Cada equipo ejecuta **dos variantes cambiando UNA sola variable** (misma seed):

| Variable | Valores |
|---|---|
| `hidden_dim` | 4 vs 64 |
| `dropout` | 0 vs 0.4 |
| `weight_decay` | 0 vs 1e-3 |
| `learning_rate` | 1e-4 vs 1e-2 |
| profundidad | 1 capa vs 4 capas |

Usa la celda siguiente como plantilla: reentrena y regenera las tres evidencias.

In [ ]:
# ── PLANTILLA DE EXPERIMENTO ─────────────────────────────────────
# 1. Declara tu hipótesis:
HIPOTESIS = 'hidden_dim=4 producirá underfitting: frontera demasiado simple.'

# 2. Cambia UNA variable:
seed_everything(42)                       # misma semilla → comparación justa
model_exp = MoonMLP(hidden_dim=4, dropout=0.10)   # ← tu cambio aquí
optimizer_exp = torch.optim.AdamW(model_exp.parameters(), lr=1e-3,
                                  weight_decay=1e-4)

model_exp, hist_exp = entrenar(
    model_exp, train_loader, val_loader, criterion, optimizer_exp, device,
    max_epochs=300, patience=20, binaria=True, verbose_cada=50,
)

# 3. Evidencia comparada — en VALIDATION, nunca en test:
# la regla de la sección 4 aplica también aquí: si comparas variantes
# contra test, ya estás usando test para elegir → benchmark overfitting.
metricas_base = run_epoch(model, val_loader, criterion, device, binaria=True)
metricas_exp = run_epoch(model_exp, val_loader, criterion, device, binaria=True)
print(f'\nHipótesis: {HIPOTESIS}')
print(f'Base      → val acc {metricas_base.accuracy:.3f}')
print(f'Variante  → val acc {metricas_exp.accuracy:.3f}')
# Solo la variante GANADORA se evalúa en test, una vez, al final.

## 7. 📝 Conclusión (máximo 150 palabras)

Estructura obligatoria: **hipótesis → evidencia → limitación → decisión**.

*Escribe aquí tu conclusión...*

---

### Entrega

- [ ] Tabla de configuración de ambos runs
- [ ] Curvas train/validation
- [ ] F1 y matriz de confusión en test
- [ ] Frontera de decisión
- [ ] Conclusión ≤ 150 palabras
- [ ] `git commit -m "feat: complete mlp experiment"` y push a tu branch